# My Approach to Tweet Preprocessing

*By Mohammad Sayem Chowdhury*

In this notebook, I share my personal workflow for preprocessing tweets, a crucial step in my sentiment analysis projects. I believe understanding and customizing each step of the pipeline helps me get the most out of my data. Here, I walk through how I use Python and NLTK to clean and prepare Twitter data for analysis.

## Getting Started

For my sentiment analysis work, I rely on the Natural Language Toolkit (NLTK) to handle and process Twitter data. NLTK provides convenient modules for collecting, cleaning, and analyzing tweets. In this notebook, I use a sample Twitter dataset included with NLTK, which is already labeled for positive and negative sentiment. This helps me quickly test and refine my preprocessing pipeline.

In [ ]:
import nltk                                # My go-to library for NLP tasks
from nltk.corpus import twitter_samples    # Sample Twitter dataset from NLTK
import matplotlib.pyplot as plt            # For visualizing data
import random                              # For selecting random samples

## About the Twitter Dataset

The NLTK sample dataset contains 5,000 positive and 5,000 negative tweets, making it perfectly balanced for testing. While real-world data is rarely this balanced, I find it useful for developing and evaluating my preprocessing steps. Later, I can adapt these methods to more complex, imbalanced datasets.

In [ ]:
# Download the sample Twitter dataset (if not already present)
nltk.download('twitter_samples')

I load the positive and negative tweets using NLTK's `strings()` method. This gives me two lists of tweets, ready for exploration and cleaning.

In [ ]:
# Load positive and negative tweets from the dataset
positive_tweets = twitter_samples.strings('positive_tweets.json')
negative_tweets = twitter_samples.strings('negative_tweets.json')

I like to check the number of tweets in each category and confirm the data structure before diving deeper.

In [ ]:
print('Number of positive tweets:', len(positive_tweets))
print('Number of negative tweets:', len(negative_tweets))

print('\nType of positive_tweets:', type(positive_tweets))
print('Type of a tweet entry:', type(negative_tweets[0]))

The tweets are stored as lists of strings. To get a quick sense of the data balance, I visualize the counts using a pie chart. This is a simple but effective way to check class distribution before moving on.

In [ ]:
# Create a pie chart to visualize class distribution
fig = plt.figure(figsize=(5, 5))
labels = ['Positive', 'Negative']
sizes = [len(positive_tweets), len(negative_tweets)]
plt.pie(sizes, labels=labels, autopct='%1.1f%%', shadow=True, startangle=90)
plt.axis('equal')  # Keep the pie chart circular
plt.show()

## Exploring Raw Tweets

Before preprocessing, I always take a look at a few sample tweets. This helps me spot common patterns, quirks, or issues that might need special handling. Here, I print a random positive and negative tweet to get a feel for the data. (Note: Tweets are real and may contain explicit content.)

In [ ]:
# Print a random positive tweet in green
print('\033[92m' + positive_tweets[random.randint(0, 4999)])

# Print a random negative tweet in red
print('\033[91m' + negative_tweets[random.randint(0, 4999)])

One observation you may have is the presence of [emoticons](https://en.wikipedia.org/wiki/Emoticon) and URLs in many of the tweets. This info will come in handy in the next steps.

## My Preprocessing Pipeline for Sentiment Analysis

For any NLP project, I find that careful data preprocessing is essential. My typical steps include:

* Tokenizing the text
* Converting to lowercase
* Removing stop words and punctuation
* Stemming words to their root form

I'll walk through each step using a sample tweet from the dataset, showing how I transform the raw text into something ready for analysis.

In [ ]:
# Select a sample tweet to demonstrate preprocessing steps
sample_tweet = positive_tweets[2277]
print(sample_tweet)

Next, I import a few more libraries to help with text cleaning and tokenization.

In [ ]:
# Download the stopwords from NLTK
nltk.download('stopwords')

In [ ]:
import re                                  # For regular expression operations
import string                              # For string operations
from nltk.corpus import stopwords          # Stop words from NLTK
from nltk.stem import PorterStemmer        # For stemming
from nltk.tokenize import TweetTokenizer   # For tokenizing strings

### Removing Twitter-Specific Text

Tweets often contain hashtags, retweet marks, and links. I use regular expressions to clean these out, making the text easier to analyze.

In [ ]:
print('\033[92m' + sample_tweet)
print('\033[94m')

# Remove retweet text "RT"
cleaned_tweet = re.sub(r'^RT[\s]+', '', sample_tweet)
# Remove hyperlinks
cleaned_tweet = re.sub(r'https?://[^\s\n\r]+', '', cleaned_tweet)
# Remove hashtags (just the # symbol)
cleaned_tweet = re.sub(r'#', '', cleaned_tweet)

print(cleaned_tweet)

### Tokenizing the Text

Tokenization splits the tweet into individual words. I also convert everything to lowercase at this stage. NLTK's TweetTokenizer makes this process straightforward.

In [ ]:
print()
print('\033[92m' + cleaned_tweet)
print('\033[94m')

# Initialize the tokenizer
my_tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True, reduce_len=True)
# Tokenize the cleaned tweet
tokens = my_tokenizer.tokenize(cleaned_tweet)

print()
print('Tokenized string:')
print(tokens)

### Removing Stop Words and Punctuation

Next, I filter out common stop words and punctuation. These words don't add much meaning and can clutter the analysis. NLTK provides a handy list of stop words, but I sometimes customize it for specific projects.

In [ ]:
# Import the English stop words list from NLTK
stopwords_english = stopwords.words('english') 

print('Stop words\n')
print(stopwords_english)

print('\nPunctuation\n')
print(string.punctuation)

Some stop words, like 'not' or 'between', might be important depending on the context. For this example, I use the full list provided by NLTK. When working with tweets, I also consider whether to keep emoticons or special punctuation, since they can carry sentiment.

In [ ]:
print()
print('\033[92m')
print(tokens)
print('\033[94m')

clean_tokens = []
for word in tokens:  # Go through every word in your tokens list
    if (word not in stopwords_english and  # remove stopwords
        word not in string.punctuation):  # remove punctuation
        clean_tokens.append(word)

print('Removed stop words and punctuation:')
print(clean_tokens)

Notice that words like **happy** and **sunny** are preserved after cleaning. This step helps focus on the most meaningful parts of each tweet.

### Stemming

Stemming reduces words to their root form, which helps group similar words together. For example, 'learning', 'learned', and 'learnt' all become 'learn'. I use NLTK's PorterStemmer for this step. Sometimes, the stemmed words aren't real words (like 'happi'), but they still help reduce vocabulary size and improve analysis.

In [ ]:
print()
print('\033[92m')
print(clean_tokens)
print('\033[94m')

# Initialize the stemmer
my_stemmer = PorterStemmer()
stemmed_tokens = []
for word in clean_tokens:
    stemmed_word = my_stemmer.stem(word)
    stemmed_tokens.append(stemmed_word)

print('Stemmed words:')
print(stemmed_tokens)

That's it! Now I have a clean set of words ready for the next stage of my sentiment analysis project.

## My process_tweet() Function

To streamline preprocessing, I use a helper function called `process_tweet()`, which combines all the steps above. You can find its implementation in my `utils.py` file. This function makes it easy to preprocess any tweet with a single call.

In [ ]:
from utils import process_tweet  # Import my custom tweet preprocessing function

# Use the same sample tweet
sample_tweet = positive_tweets[2277]

print()
print('\033[92m')
print(sample_tweet)
print('\033[94m')

# Call my helper function
tweet_processed = process_tweet(sample_tweet)

print('Preprocessed tweet:')
print(tweet_processed)

Thank you for following along with my tweet preprocessing workflow! I hope this gives you insight into my approach and inspires you to customize your own pipeline. 

*Notebook by Mohammad Sayem Chowdhury, June 2025*